In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "c0",
   "metadata": {},
   "source": [
    "# CineMatch: ML analysis\n",
    "\n",
    "Charts and numbers for the project report. Run all cells top to bottom (**Run All**).\n",
    "Data comes from `../backend/movies.csv` and `../backend/ratings.csv` (synthetic sample ratings).\n",
    "\n",
    "Sections: 1. Data overview, 2. Unsupervised (K-Means + PCA), 3. Supervised (rating prediction), 4. Advanced (SVD + neural network), 5. Summary."
   ]
  },
  {
   "cell_type": "code",
   "id": "c1",
   "metadata": {},
   "source": [
    "import warnings\n",
    "import numpy as np, pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "from sklearn.cluster import KMeans\n",
    "from sklearn.decomposition import PCA, TruncatedSVD\n",
    "from sklearn.ensemble import RandomForestRegressor\n",
    "from sklearn.linear_model import LinearRegression\n",
    "from sklearn.neighbors import KNeighborsRegressor\n",
    "from sklearn.neural_network import MLPRegressor\n",
    "from sklearn.pipeline import make_pipeline\n",
    "from sklearn.preprocessing import StandardScaler, normalize\n",
    "from sklearn.metrics import mean_squared_error, mean_absolute_error, silhouette_score\n",
    "from sklearn.model_selection import train_test_split\n",
    "\n",
    "warnings.filterwarnings(\"ignore\")\n",
    "plt.rcParams.update({\"figure.figsize\": (7, 4), \"axes.spines.top\": False, \"axes.spines.right\": False, \"axes.grid\": True, \"grid.alpha\": .25})\n",
    "RED = \"#e50914\"\n",
    "GENRES = [\"Action\", \"Sci-Fi\", \"Romance\", \"Comedy\", \"Drama\", \"Thriller\", \"Family\", \"Horror\"]\n",
    "rmse = lambda a, b: mean_squared_error(a, b) ** 0.5"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "id": "c2",
   "metadata": {},
   "source": [
    "## 1. Data overview"
   ]
  },
  {
   "cell_type": "code",
   "id": "c3",
   "metadata": {},
   "source": [
    "movies = pd.read_csv(\"../backend/movies.csv\").sort_values(\"id\").reset_index(drop=True)\n",
    "ratings = pd.read_csv(\"../backend/ratings.csv\")\n",
    "G = movies[GENRES].values.astype(float)\n",
    "R = ratings.pivot_table(index=\"user_id\", columns=\"movie_id\", values=\"rating\", fill_value=0).reindex(columns=movies[\"id\"], fill_value=0)\n",
    "print(f\"{len(movies)} movies, {R.shape[0]} users, {len(ratings)} ratings\")\n",
    "print(f\"Matrix sparsity: {(R.values == 0).mean():.0%} empty\")\n",
    "movies.head()"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "code",
   "id": "c4",
   "metadata": {},
   "source": [
    "# Rating histogram\n",
    "counts = ratings[\"rating\"].value_counts().sort_index()\n",
    "(counts / counts.sum() * 100).plot(kind=\"bar\", color=RED)\n",
    "plt.xlabel(\"Stars\"); plt.ylabel(\"% of ratings\"); plt.title(\"Rating distribution\"); plt.xticks(rotation=0)\n",
    "plt.show()"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "code",
   "id": "c5",
   "metadata": {},
   "source": [
    "# Movies per genre\n",
    "pd.Series(G.sum(0), index=GENRES).sort_values().plot(kind=\"barh\", color=RED)\n",
    "plt.xlabel(\"Number of movies\"); plt.title(\"Movies per genre\")\n",
    "plt.show()"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "id": "c6",
   "metadata": {},
   "source": [
    "## 2. Unsupervised learning: K-Means user clustering\n",
    "\n",
    "Each user is described by a genre-taste vector built from their ratings. The elbow and silhouette charts show why **k = 4** is a good choice."
   ]
  },
  {
   "cell_type": "code",
   "id": "c7",
   "metadata": {},
   "source": [
    "UT = np.clip(R.values - 2.5, 0, None) @ G      # user taste per genre\n",
    "Xu = normalize(UT)\n",
    "\n",
    "inertia = [KMeans(k, n_init=10, random_state=42).fit(Xu).inertia_ for k in range(1, 9)]\n",
    "sil = {k: silhouette_score(Xu, KMeans(k, n_init=10, random_state=42).fit_predict(Xu)) for k in range(2, 9)}\n",
    "\n",
    "fig, ax = plt.subplots(1, 2, figsize=(11, 4))\n",
    "ax[0].plot(range(1, 9), inertia, marker=\"o\", color=RED); ax[0].axvline(4, ls=\"--\", color=\"gray\")\n",
    "ax[0].set(xlabel=\"k (clusters)\", ylabel=\"Inertia\", title=\"Elbow method\")\n",
    "ax[1].plot(list(sil), list(sil.values()), marker=\"o\", color=\"#6ea8ff\"); ax[1].axvline(4, ls=\"--\", color=\"gray\")\n",
    "ax[1].set(xlabel=\"k (clusters)\", ylabel=\"Silhouette score\", title=\"Silhouette score\")\n",
    "plt.tight_layout(); plt.show()\n",
    "print(f\"Silhouette at k=4: {sil[4]:.2f}\")"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "code",
   "id": "c8",
   "metadata": {},
   "source": [
    "km = KMeans(4, n_init=10, random_state=42).fit(Xu)\n",
    "pca = PCA(2, random_state=42).fit(Xu)\n",
    "Z, C = pca.transform(Xu), pca.transform(km.cluster_centers_)\n",
    "\n",
    "plt.figure(figsize=(7, 5))\n",
    "for c in range(4):\n",
    "    m = km.labels_ == c\n",
    "    top = [GENRES[i] for i in np.argsort(-km.cluster_centers_[c])[:2]]\n",
    "    plt.scatter(Z[m, 0], Z[m, 1], s=16, alpha=.7, label=f\"Cluster {c}: {', '.join(top)} ({m.sum()})\")\n",
    "plt.scatter(C[:, 0], C[:, 1], c=\"black\", marker=\"X\", s=150, label=\"Centres\")\n",
    "plt.xlabel(\"PC1\"); plt.ylabel(\"PC2\"); plt.title(\"User clusters (PCA 2D)\"); plt.legend(fontsize=8)\n",
    "plt.show()\n",
    "print(f\"PCA explains {pca.explained_variance_ratio_.sum():.0%} of the variance in 2D\")"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "id": "c9",
   "metadata": {},
   "source": [
    "## 3. Supervised learning: rating prediction\n",
    "\n",
    "Inputs: genre vector, year, average rating and the user's genre affinity. Output: the star rating. 80% of ratings train the model, 20% test it."
   ]
  },
  {
   "cell_type": "code",
   "id": "c10",
   "metadata": {},
   "source": [
    "def cos_rows(a, b):\n",
    "    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-9)\n",
    "\n",
    "uidx = R.index.get_indexer(ratings[\"user_id\"]); mid = ratings[\"movie_id\"].values\n",
    "affinity = cos_rows(UT[uidx], G[mid])\n",
    "names = GENRES + [\"year\", \"avg rating\", \"affinity\"]\n",
    "X = np.column_stack([G[mid], movies[\"year\"].values[mid], movies[\"avg\"].values[mid], affinity])\n",
    "y = ratings[\"rating\"].values\n",
    "Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42)\n",
    "\n",
    "models = {\"LinReg\": LinearRegression(), \"KNN\": KNeighborsRegressor(15), \"RandomForest\": RandomForestRegressor(200, random_state=42)}\n",
    "results, preds = {}, {}\n",
    "for n, m in models.items():\n",
    "    m.fit(Xtr, ytr); p = np.clip(m.predict(Xte), 1, 5)\n",
    "    results[n] = (rmse(yte, p), mean_absolute_error(yte, p)); preds[n] = p\n",
    "for n, (r, a) in results.items(): print(f\"{n:13s} RMSE {r:.3f}   MAE {a:.3f}\")"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "code",
   "id": "c11",
   "metadata": {},
   "source": [
    "fig, ax = plt.subplots(1, 2, figsize=(11, 4))\n",
    "ax[0].bar(results.keys(), [v[0] for v in results.values()], color=\"#46d369\")\n",
    "ax[0].set(ylabel=\"RMSE (lower is better)\", title=\"Supervised model comparison\")\n",
    "rng = np.random.default_rng(0)\n",
    "ax[1].scatter(yte + rng.normal(0, .08, len(yte)), preds[\"RandomForest\"], s=8, alpha=.4, color=\"#46d369\")\n",
    "ax[1].plot([1, 5], [1, 5], \"--\", color=RED); ax[1].set(xlabel=\"Actual rating\", ylabel=\"Predicted rating\", title=\"Random Forest: actual vs predicted\")\n",
    "plt.tight_layout(); plt.show()"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "code",
   "id": "c12",
   "metadata": {},
   "source": [
    "imp = pd.Series(models[\"RandomForest\"].feature_importances_, index=names).sort_values()\n",
    "imp.plot(kind=\"barh\", color=RED); plt.xlabel(\"Importance\"); plt.title(\"Random Forest feature importance\")\n",
    "plt.show()"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "id": "c13",
   "metadata": {},
   "source": [
    "## 4. Advanced ML: SVD matrix factorization and a neural network\n",
    "\n",
    "SVD learns hidden taste factors from the user-movie matrix. A small neural network (MLP) is used as a simple neural collaborative model."
   ]
  },
  {
   "cell_type": "code",
   "id": "c14",
   "metadata": {},
   "source": [
    "tr_i, te_i = train_test_split(np.arange(len(ratings)), test_size=.2, random_state=42)\n",
    "tr, te = ratings.iloc[tr_i], ratings.iloc[te_i]\n",
    "\n",
    "M = tr.pivot_table(index=\"user_id\", columns=\"movie_id\", values=\"rating\").reindex(index=R.index, columns=movies[\"id\"])\n",
    "um = M.mean(axis=1).fillna(tr[\"rating\"].mean()).values[:, None]\n",
    "Mc = M.fillna(pd.DataFrame(np.repeat(um, M.shape[1], 1), index=M.index, columns=M.columns)).values - um\n",
    "svd = TruncatedSVD(8, random_state=42).fit(Mc)\n",
    "recon = svd.inverse_transform(svd.transform(Mc)) + um\n",
    "\n",
    "ui = R.index.get_indexer(te[\"user_id\"]); mi = te[\"movie_id\"].values\n",
    "p_svd = np.clip(recon[ui, mi], 1, 5)\n",
    "results[\"SVD\"] = (rmse(te[\"rating\"], p_svd), mean_absolute_error(te[\"rating\"], p_svd))\n",
    "\n",
    "mlp = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42)).fit(Xtr, ytr)\n",
    "p_mlp = np.clip(mlp.predict(Xte), 1, 5)\n",
    "results[\"Neural (MLP)\"] = (rmse(yte, p_mlp), mean_absolute_error(yte, p_mlp))\n",
    "print(f\"SVD          RMSE {results['SVD'][0]:.3f}\\nNeural (MLP) RMSE {results['Neural (MLP)'][0]:.3f}\")\n",
    "\n",
    "plt.plot(mlp[-1].loss_curve_, color=RED); plt.xlabel(\"Epoch\"); plt.ylabel(\"Training loss\"); plt.title(\"Neural network training curve\")\n",
    "plt.show()"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "code",
   "id": "c15",
   "metadata": {},
   "source": [
    "# Precision@K: hybrid-style SVD ranking vs a \"most popular\" baseline\n",
    "seen = M.notna().values\n",
    "rel = te[te[\"rating\"] >= 4].groupby(\"user_id\")[\"movie_id\"].apply(set)\n",
    "\n",
    "def precision_at_k(scores, k):\n",
    "    hits = []\n",
    "    for u, items in rel.items():\n",
    "        s = np.where(seen[R.index.get_loc(u)], -np.inf, scores[R.index.get_loc(u)])\n",
    "        hits.append(len(items & set(np.argsort(-s)[:k])) / k)\n",
    "    return float(np.mean(hits))\n",
    "\n",
    "pop = np.tile(movies[\"avg\"].values, (len(R), 1))\n",
    "ks = [3, 5, 10]\n",
    "pk = pd.DataFrame({\"Popularity\": [precision_at_k(pop, k) for k in ks], \"SVD\": [precision_at_k(recon, k) for k in ks]}, index=[f\"P@{k}\" for k in ks])\n",
    "pk.plot(kind=\"bar\", color=[\"#777\", RED]); plt.ylabel(\"Precision\"); plt.title(\"Precision@K: SVD vs popularity baseline\"); plt.xticks(rotation=0)\n",
    "plt.show()\n",
    "pk.round(3)"
   ],
   "outputs": [],
   "execution_count": null
  },
  {
   "cell_type": "markdown",
   "id": "c16",
   "metadata": {},
   "source": [
    "## 5. Summary\n",
    "\n",
    "Copy these numbers into your report and into the ML lab tab of the frontend."
   ]
  },
  {
   "cell_type": "code",
   "id": "c17",
   "metadata": {},
   "source": [
    "summary = pd.DataFrame(results, index=[\"RMSE\", \"MAE\"]).T.round(3).sort_values(\"RMSE\")\n",
    "summary.plot(kind=\"barh\", y=\"RMSE\", color=RED, legend=False); plt.xlabel(\"RMSE (lower is better)\"); plt.title(\"All models\")\n",
    "plt.show()\n",
    "summary"
   ],
   "outputs": [],
   "execution_count": null
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}